# C-STDP Interpretability Trace

This notebook dives into the mechanism of *why* a specific link was inferred.

We will trace the spike pairs for a single Ground Truth edge (Source -> Target) and visualize how the weight evolves.

In [ ]:
import sys
sys.path.append("../")
import numpy as np
import matplotlib.pyplot as plt
from src.cstdp.utils.simulate_grn import generate_synthetic_grn, simulate_expression
from src.cstdp.utils.spike_encoding import calculate_adaptive_thresholds
from src.cstdp.core import CausalSTDP

## Setup Single Pair Experiment

In [ ]:
# Generate data
np.random.seed(42)
n_genes = 5
true_adj, delays = generate_synthetic_grn(n_genes, connection_prob=0.3)
expression = simulate_expression(n_genes, 1000, true_adj, delays)

# Encode
thresholds = calculate_adaptive_thresholds(expression, sigma=1.5)
cstdp = CausalSTDP(A_pos=0.05, A_neg=0.06)
spike_trains = cstdp.compute_spike_times(expression, np.arange(1000), thresholds)

# Find a valid connection to trace
sources, targets = np.where(true_adj > 0)
if len(sources) > 0:
    src_idx = sources[0]
    tgt_idx = targets[0]
    print(f"Tracing connection: Gene {src_idx} -> Gene {tgt_idx}")
else:
    print("No connections found in random graph.")

## Trace Weight Evolution

In [ ]:
if len(sources) > 0:
    spikes_src = spike_trains[src_idx]
    spikes_tgt = spike_trains[tgt_idx]
    
    print(f"Spikes Src: {len(spikes_src)}")
    print(f"Spikes Tgt: {len(spikes_tgt)}")
    
    # Manually calculate weight updates
    weight_trace = [0.0]
    times = [0.0]
    current_w = 0.0
    
    # We iterate through time to show evolution
    # Simplification: Just listing pairs
    
    updates = []
    for t_s in spikes_src:
        for t_t in spikes_tgt:
            dw = cstdp.stdp_update(0, t_s, t_t)
            if abs(dw) > 1e-5:
                updates.append((max(t_s, t_t), dw))
                
    # Sort by time of occurrence (approx)
    updates.sort(key=lambda x: x[0])
    
    for t, dw in updates:
        current_w += dw
        current_w = np.clip(current_w, 0, 1.0)
        weight_trace.append(current_w)
        times.append(t)
        
    plt.figure(figsize=(10, 4))
    plt.step(times, weight_trace)
    plt.title(f"Weight Evolution: {src_idx} -> {tgt_idx}")
    plt.xlabel("Time")
    plt.ylabel("Weight")
    plt.show()